In [21]:
from langgraph.graph import StateGraph,START,END
from langgraph.types import Command,interrupt
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from llm import llm

In [22]:
class State(TypedDict):
    topic:str
    draft:str
    approved:bool


In [23]:
def write(state:State):
    return {"draft":llm.invoke(f"Write a funny mail on topic : {state['topic']}").content}

def review(state:State):
    decision=interrupt({
        'draft':state['draft'],
        "send":"Send this email or not (yes or no )?"
    })
    print("---------------",decision)
    return {"approved":decision.lower()=="yes"}

def router(state:State):
    return state["approved"]==True
def send(state:State):
    print("Sent successfully")
    return {}
def discard(state:State):
    print("Sent discarded")
    return {}

In [24]:
graph=StateGraph(State)
config={"configurable":{"thread_id":"1"}}
checkpoint=InMemorySaver()

graph.add_node("write",write)
graph.add_node("review",review)
graph.add_node("send",send)
graph.add_node("discard",discard)


graph.add_edge(START,"write")
graph.add_edge("write","review")
graph.add_conditional_edges("review",router,{True:'send',False:'discard'})
graph.add_edge("send",END)
graph.add_edge("discard",END)

workflow=graph.compile(checkpointer=checkpoint)


In [29]:
workflow.invoke({"topic":"Dev"},config=config)

a=input("Y/N")

workflow.invoke(Command(resume=a),config=config)

--------------- yes
Sent successfully


{'topic': 'Dev', 'draft': '', 'approved': True}

TypeError: interrupt() takes 1 positional argument but 2 were given